In [ ]:
!pip install osmnx networkx folium matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.0 MB/s eta 0:00:00


In [ ]:
PONTO_A = {"nome": "Museu de Morfologia", "lat": -5.841777929488095, "lon": -35.203041910059596}
PONTO_B = {"nome": "Assai Atacadista",  "lat": -5.858685640917017, "lon": -35.195926594143344}

X = 500  # distância máxima de caminhada em metros

print(f"A: {PONTO_A['nome']}")
print(f"B: {PONTO_B['nome']}")
print(f"Caminhada máxima: {X} m")

A: Museu de Morfologia
B: Assai Atacadista
Caminhada máxima: 500 m


In [ ]:
import osmnx as ox
import networkx as nx

centro = ((PONTO_A["lat"] + PONTO_B["lat"]) / 2,
          (PONTO_A["lon"] + PONTO_B["lon"]) / 2)

# Grafo de CARRO — usado para o trecho P → B
G = ox.graph_from_point(centro, dist=4000, network_type="drive")
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

# Grafo de PEDESTRE — usado apenas para encontrar candidatos P
# e calcular a distância/tempo de caminhada A → P
# Raio menor: só precisa cobrir X metros ao redor de A
G_walk = ox.graph_from_point(
    (PONTO_A["lat"], PONTO_A["lon"]),
    dist=X + 200,           # margem extra de 200m para não cortar candidatos na borda
    network_type="walk"
)

# Nó de A em cada grafo
no_A      = ox.distance.nearest_nodes(G,      PONTO_A["lon"], PONTO_A["lat"])
no_A_walk = ox.distance.nearest_nodes(G_walk, PONTO_A["lon"], PONTO_A["lat"])

# Nó de B apenas no grafo de carro
no_B = ox.distance.nearest_nodes(G, PONTO_B["lon"], PONTO_B["lat"])

print(f"Grafo drive  — Nós: {G.number_of_nodes()} | Arestas: {G.number_of_edges()}")
print(f"Grafo walk   — Nós: {G_walk.number_of_nodes()} | Arestas: {G_walk.number_of_edges()}")
print(f"Nó A (drive): {no_A} | Nó A (walk): {no_A_walk} | Nó B: {no_B}")

Grafo drive  — Nós: 4965 | Arestas: 12359
Grafo walk   — Nós: 938 | Arestas: 2624
Nó A (drive): 501834722 | Nó A (walk): 6956477611 | Nó B: 503296428


In [ ]:
import random
import copy

random.seed(42)

# Candidatos P: exclui o próprio A — P deve ser um ponto DIFERENTE de A
candidatos = nx.single_source_dijkstra_path_length(
    G_walk, no_A_walk, cutoff=X, weight="length"
)
candidatos = {n: d for n, d in candidatos.items() if n != no_A_walk}

print(f"Candidatos a P (≤ {X} m, rede pedestre): {len(candidatos)} nós")

# Trânsito sintético: aplicado apenas ao grafo de CARRO
G_tr = copy.deepcopy(G)
for u, v, data in G_tr.edges(data=True):
    data["travel_time_transito"] = data.get("travel_time", 1.0) * random.uniform(1.0, 3.5)

print("Grafo com trânsito sintético pronto.")

Candidatos a P (≤ 500 m, rede pedestre): 82 nós
Grafo com trânsito sintético pronto.


In [ ]:
import math
import random

random.seed(42)

VELOCIDADE_PEDESTRE = 1.4  # m/s

def avaliar_P(G, no_P, no_B, dist_walk, dijkstra_func, peso_tempo="travel_time"):
    """Calcula distância total e tempo total (caminhada + carro) para um dado P."""
    carro_dist,  _ = dijkstra_func(G, no_P, no_B, weight="length")
    carro_tempo, _ = dijkstra_func(G, no_P, no_B, weight=peso_tempo)

    tempo_walk = dist_walk / VELOCIDADE_PEDESTRE

    return {
        "no_P": no_P,
        "walk_m": dist_walk,
        "dist_total_m": dist_walk + carro_dist,
        "tempo_total_s": tempo_walk + carro_tempo,
    }

def melhor_P(G, candidatos, no_B, criterio, dijkstra_func, peso_tempo="travel_time"):
    melhor_custo, melhor_info = math.inf, None
    for no_P, dist_walk in candidatos.items():
        if no_P not in G.nodes:
            continue
        info = avaliar_P(G, no_P, no_B, dist_walk, dijkstra_func, peso_tempo)
        if info[criterio] < melhor_custo:
            melhor_custo = info[criterio]
            melhor_info  = info
    return melhor_info


def cenario4_sem_caminhada(G, G_walk, no_A_walk, no_A_drive, candidatos,
                            no_B, dijkstra_func):
    """
    Cenário 4 — Sem caminhada (parte do ponto A exato se possível).

    Regra:
      - Se o nó snap de A no grafo drive coincide com as coordenadas exatas
        de PONTO_A (i.e., o ponto A está literalmente no grafo de carro),
        parte diretamente de no_A_drive sem qualquer caminhada.
      - Caso contrário, caminha até o nó do grafo drive MAIS PRÓXIMO
        (menor distância geográfica via G_walk), contabilizando esse
        trecho como caminhada na distância e no tempo totais.

    Retorna um dicionário compatível com avaliar_P:
      {no_P, walk_m, dist_total_m, tempo_total_s, c4_caminhou}
    """
    import osmnx as ox

    # Coordenadas do nó snap de A no grafo drive
    snap_lat = G.nodes[no_A_drive]["y"]
    snap_lon = G.nodes[no_A_drive]["x"]

    # Distância geográfica entre o ponto A real e o nó snap
    dist_snap = ox.distance.great_circle(
        PONTO_A["lat"], PONTO_A["lon"], snap_lat, snap_lon
    )

    TOLERANCIA_M = 1.0  # considera "mesmo ponto" se < 1 metro de diferença

    if dist_snap < TOLERANCIA_M:
        # A exato está no grafo drive — sem caminhada
        no_P       = no_A_drive
        walk_m     = 0.0
        caminhou   = False
    else:
        # A não está no grafo drive — encontra o P mais próximo (sem otimizar)
        # Usa os candidatos já calculados na rede pedestre; o mais próximo é
        # aquele com menor dist_walk que também exista no grafo de carro.
        no_P_mais_proximo = None
        menor_dist        = math.inf
        for no_P_cand, dist_walk_cand in candidatos.items():
            if no_P_cand in G.nodes and dist_walk_cand < menor_dist:
                menor_dist        = dist_walk_cand
                no_P_mais_proximo = no_P_cand

        if no_P_mais_proximo is None:
            # Fallback: usa o nearest_node padrão do osmnx (sem caminhada na rede)
            no_P     = no_A_drive
            walk_m   = dist_snap
            caminhou = True
        else:
            no_P     = no_P_mais_proximo
            walk_m   = menor_dist
            caminhou = True

    carro_dist,  _ = dijkstra_func(G, no_P, no_B, weight="length")
    carro_tempo, _ = dijkstra_func(G, no_P, no_B, weight="travel_time")
    tempo_walk     = walk_m / VELOCIDADE_PEDESTRE

    return {
        "no_P":          no_P,
        "walk_m":        walk_m,
        "dist_total_m":  walk_m + carro_dist,
        "tempo_total_s": tempo_walk + carro_tempo,
        "c4_caminhou":   caminhou,
    }


def calcular_5_cenarios(G, G_tr, G_walk, candidatos, no_A_walk, no_A_drive,
                         no_B, dijkstra_func):
    # Cenários 1, 2, 3 — busca pelo melhor P ≠ A
    c1 = melhor_P(G,    candidatos, no_B, "dist_total_m",  dijkstra_func)
    c2 = melhor_P(G,    candidatos, no_B, "tempo_total_s", dijkstra_func)
    c3 = melhor_P(G_tr, candidatos, no_B, "tempo_total_s", dijkstra_func,
                  peso_tempo="travel_time_transito")

    # Cenário 4 — sem caminhada (parte de A exato ou do P mais próximo)
    c4 = cenario4_sem_caminhada(G, G_walk, no_A_walk, no_A_drive,
                                 candidatos, no_B, dijkstra_func)
    d4_dist  = c4["dist_total_m"]
    d4_tempo = c4["tempo_total_s"]

    # Cenário 5 — ganho ao caminhar (pode ser negativo: significa que não vale a pena)
    ganho_dist  = d4_dist  - c1["dist_total_m"]
    ganho_tempo = d4_tempo - c2["tempo_total_s"]

    # Rótulo do cenário 4 adaptado conforme se houve caminhada forçada
    if c4["c4_caminhou"]:
        label_c4 = f"4 – Sem caminhada (A∉drive, walk {c4['walk_m']:.0f} m)"
    else:
        label_c4 = "4 – Sem caminhada (A→B direto)"

    return [
        ("1 – Menor distância (c/ P)",        c1["dist_total_m"], c1["tempo_total_s"]),
        ("2 – Mais rápida s/ trânsito (c/P)", c2["dist_total_m"], c2["tempo_total_s"]),
        ("3 – Mais rápida c/ trânsito (c/P)", c3["dist_total_m"], c3["tempo_total_s"]),
        (label_c4,                             d4_dist,            d4_tempo),
        ("5 – Ganho ao caminhar (vs cen.4)",  ganho_dist,         ganho_tempo),
    ], (c1, c2, c3, c4)


def imprimir_cenarios(titulo, cenarios):
    print(f"\n=== {titulo} ===")
    print(f"{'Cenário':<52} {'Distância (m)':>14} {'Tempo (s)':>12}")
    print("─" * 82)
    for label, dist, tempo in cenarios:
        d = f"{dist:.1f}"  if isinstance(dist,  float) else dist
        t = f"{tempo:.1f}" if isinstance(tempo, float) else tempo
        print(f"{label:<52} {d:>14} {t:>12}")


# ── Dijkstra Bidirecional: testa TODOS os candidatos (é rápido o suficiente) ───────
cenarios_heap, (c1, c2, c3, c4) = calcular_5_cenarios(
    G, G_tr, G_walk, candidatos, no_A_walk, no_A, no_B, nx.bidirectional_dijkstra
)
imprimir_cenarios("Dijkstra Heap (todos os candidatos)", cenarios_heap)

print(f"\nP ótimo (distância)        : nó {c1['no_P']} — caminhada {c1['walk_m']:.0f} m"
      f"{' (= A, sem caminhar)' if c1['walk_m'] == 0 else ''}")
print(f"P ótimo (tempo s/ trânsito) : nó {c2['no_P']} — caminhada {c2['walk_m']:.0f} m"
      f"{' (= A, sem caminhar)' if c2['walk_m'] == 0 else ''}")
print(f"P ótimo (tempo c/ trânsito) : nó {c3['no_P']} — caminhada {c3['walk_m']:.0f} m"
      f"{' (= A, sem caminhar)' if c3['walk_m'] == 0 else ''}")
if c4["c4_caminhou"]:
    print(f"C4 (sem caminhada)          : A NÃO está no grafo drive → caminhou "
          f"{c4['walk_m']:.0f} m até nó {c4['no_P']} (P mais próximo)")
else:
    print(f"C4 (sem caminhada)          : A está no grafo drive → partiu direto do nó {c4['no_P']}")



=== Dijkstra Heap (todos os candidatos) ===
Cenário                                               Distância (m)    Tempo (s)
──────────────────────────────────────────────────────────────────────────────────
1 – Menor distância (c/ P)                                   2990.1        412.3
2 – Mais rápida s/ trânsito (c/P)                            3238.4        320.9
3 – Mais rápida c/ trânsito (c/P)                            3238.4        626.2
4 – Sem caminhada (A∉drive, walk 144 m)                      3238.4        320.9
5 – Ganho ao caminhar (vs cen.4)                              248.2          0.0

P ótimo (distância)        : nó 501033902 — caminhada 250 m
P ótimo (tempo s/ trânsito) : nó 500986659 — caminhada 144 m
P ótimo (tempo c/ trânsito) : nó 500986659 — caminhada 144 m
C4 (sem caminhada)          : A NÃO está no grafo drive → caminhou 144 m até nó 500986659 (P mais próximo)


In [ ]:
imprimir_cenarios("Dijkstra Bidirecional (todos os candidatos)", cenarios_heap)

print(f"\nP ótimo (distância)        : nó {c1['no_P']} — caminhada {c1['walk_m']:.0f} m")
print(f"P ótimo (tempo s/ trânsito) : nó {c2['no_P']} — caminhada {c2['walk_m']:.0f} m")
print(f"P ótimo (tempo c/ trânsito) : nó {c3['no_P']} — caminhada {c3['walk_m']:.0f} m")
if c4["c4_caminhou"]:
    print(f"C4 (sem caminhada) : A NÃO está no grafo drive → caminhou {c4['walk_m']:.0f} m até nó {c4['no_P']}")
else:
    print(f"C4 (sem caminhada) : A está no grafo drive → partiu direto do nó {c4['no_P']}")


=== Dijkstra Bidirecional (todos os candidatos) ===
Cenário                                               Distância (m)    Tempo (s)
──────────────────────────────────────────────────────────────────────────────────
1 – Menor distância (c/ P)                                   2990.1        412.3
2 – Mais rápida s/ trânsito (c/P)                            3238.4        320.9
3 – Mais rápida c/ trânsito (c/P)                            3238.4        626.2
4 – Sem caminhada (A∉drive, walk 144 m)                      3238.4        320.9
5 – Ganho ao caminhar (vs cen.4)                              248.2          0.0

P ótimo (distância)        : nó 501033902 — caminhada 250 m
P ótimo (tempo s/ trânsito) : nó 500986659 — caminhada 144 m
P ótimo (tempo c/ trânsito) : nó 500986659 — caminhada 144 m
C4 (sem caminhada) : A NÃO está no grafo drive → caminhou 144 m até nó 500986659


In [ ]:
# Célula de diagnóstico — comparar os P escolhidos por cada critério

def diagnostico_cenarios(c1, c2, c3, no_A):
    print("=== Diagnóstico: P escolhido por critério ===\n")

    info = [
        ("Cenário 1 (menor distância)",        c1),
        ("Cenário 2 (mais rápido s/ trânsito)", c2),
        ("Cenário 3 (mais rápido c/ trânsito)", c3),
    ]

    for label, c in info:
        eh_A = "← é o próprio A (sem caminhar)" if c["no_P"] == no_A else ""
        print(f"{label:<38} P = {c['no_P']:<12} caminhada = {c['walk_m']:>6.0f} m {eh_A}")

    print()
    # Verifica se cenário 1 e 2 escolheram o mesmo P
    if c1["no_P"] == c2["no_P"]:
        print(f"✓ Cenários 1 e 2 escolheram o MESMO P (nó {c1['no_P']}).")
        print("  → Por isso a distância é igual. O tempo pode variar só se houver")
        print("    rotas alternativas com mesma distância e velocidades diferentes,")
        print("    ou pode ser idêntico também (mesma rota exata).")
    else:
        print(f"✗ Cenários 1 e 2 escolheram P DIFERENTES "
              f"({c1['no_P']} vs {c2['no_P']}).")
        print("  → Isso é esperado: o ponto de menor distância nem sempre")
        print("    é o de menor tempo (ruas mais rápidas podem ser mais longas).")

    print()
    if c1["no_P"] == c2["no_P"]:
        # Mesmo P — verifica se a rota completa (lista de nós) também é igual
        _, rota1_dist = nx.bidirectional_dijkstra(G, c1["no_P"], no_B, weight="length")
        _, rota2_tempo = nx.bidirectional_dijkstra(G, c2["no_P"], no_B, weight="travel_time")
        if rota1_dist == rota2_tempo:
            print("✓ A rota P→B também é EXATAMENTE a mesma sequência de nós.")
            print("  → Distância e tempo vêm do mesmo caminho físico.")
        else:
            print("⚠ Mesmo P, mas rotas P→B são DIFERENTES sequências de nós.")
            print(f"  Rota por distância: {len(rota1_dist)} nós")
            print(f"  Rota por tempo    : {len(rota2_tempo)} nós")
            print("  → Existem 2+ caminhos com a mesma distância total,")
            print("    mas um percorre ruas mais rápidas (velocidade maior).")


diagnostico_cenarios(c1, c2, c3, no_A)

=== Diagnóstico: P escolhido por critério ===

Cenário 1 (menor distância)            P = 501033902    caminhada =    250 m 
Cenário 2 (mais rápido s/ trânsito)    P = 500986659    caminhada =    144 m 
Cenário 3 (mais rápido c/ trânsito)    P = 500986659    caminhada =    144 m 

✗ Cenários 1 e 2 escolheram P DIFERENTES (501033902 vs 500986659).
  → Isso é esperado: o ponto de menor distância nem sempre
    é o de menor tempo (ruas mais rápidas podem ser mais longas).



In [ ]:
print("Comparação P escolhidos:\n")
print(f"P (menor distância) = {c1['no_P']}")
print(f"  Distância total: {c1['dist_total_m']:.1f} m")
print(f"  Tempo total    : {c1['tempo_total_s']:.1f} s\n")

print(f"P (menor tempo)      = {c2['no_P']} (= A)")
print(f"  Distância total: {c2['dist_total_m']:.1f} m")
print(f"  Tempo total    : {c2['tempo_total_s']:.1f} s\n")

if c1["dist_total_m"] < c2["dist_total_m"] and c1["tempo_total_s"] > c2["tempo_total_s"]:
    print("✓ Faz sentido: o P de menor distância exige caminhar e/ou usa ruas mais")
    print("  lentas de carro, então mesmo sendo mais curto, é mais devagar.")
elif c1["tempo_total_s"] < c2["tempo_total_s"]:
    print("⚠ Inconsistente: o P de 'menor distância' tem tempo MENOR que o P de")
    print("  'menor tempo' — isso seria um bug, já que o cenário 2 deveria ter")
    print("  encontrado esse resultado primeiro.")

Comparação P escolhidos:

P (menor distância) = 501033902
  Distância total: 2990.1 m
  Tempo total    : 412.3 s

P (menor tempo)      = 500986659 (= A)
  Distância total: 3238.4 m
  Tempo total    : 320.9 s

✓ Faz sentido: o P de menor distância exige caminhar e/ou usa ruas mais
  lentas de carro, então mesmo sendo mais curto, é mais devagar.


In [ ]:
print("=== Verificação detalhada ===\n")

# Recalcula cada trecho separadamente, sem usar os valores já salvos em c1h/c2h
P1 = c1["no_P"]
walk_P1 = candidatos.get(P1, 0.0)  # distância de caminhada A→P1
carro_P1_B, caminho_P1_B = nx.bidirectional_dijkstra(G, P1, no_B, weight="length")

dist_A_B, caminho_A_B = nx.bidirectional_dijkstra(G, no_A, no_B, weight="length")

print(f"Caminhada A→P1 ({P1}): {walk_P1:.1f} m")
print(f"Carro     P1→B       : {carro_P1_B:.1f} m")
print(f"Soma (P1)             : {walk_P1 + carro_P1_B:.1f} m\n")

print(f"Carro A→B direto      : {dist_A_B:.1f} m\n")

# Checa se P1 está literalmente NO caminho de A até B
if P1 in caminho_A_B:
    posicao = caminho_A_B.index(P1)
    print(f"✓ P1 está no caminho A→B direto, na posição {posicao} de {len(caminho_A_B)} nós.")
    print("  → Isso explica a coincidência: caminhar 'corta' até um ponto que")
    print("    já fica no trajeto, então a soma das partes bate com o todo.")
else:
    print("⚠ P1 NÃO está no caminho A→B direto — a coincidência de distância")
    print("  pode ser apenas numérica (duas rotas físicas diferentes com soma igual),")
    print("  ou pode haver inconsistência nos pesos usados. Vale investigar mais.")

=== Verificação detalhada ===

Caminhada A→P1 (501033902): 250.3 m
Carro     P1→B       : 2739.8 m
Soma (P1)             : 2990.1 m

Carro A→B direto      : 2632.6 m

⚠ P1 NÃO está no caminho A→B direto — a coincidência de distância
  pode ser apenas numérica (duas rotas físicas diferentes com soma igual),
  ou pode haver inconsistência nos pesos usados. Vale investigar mais.


In [ ]:
import time
import pandas as pd

linhas = []
for peso, label in [("length", "Distância"), ("travel_time", "Tempo")]:
    t0 = time.perf_counter(); nx.bidirectional_dijkstra(G, no_A, no_B, weight=peso); ts = time.perf_counter() - t0
    linhas.append({"Peso": label, "Tempo (s)": ts})


In [ ]:
import folium

def coords_rota(G, caminho):
    """Converte uma lista de nós em lista de coordenadas (lat, lon) para o Folium."""
    return [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in caminho]

# ── Caminhos de CAMINHADA A→P para cada cenário (rede de pedestre) ─────
_, rota_walk_c1 = nx.bidirectional_dijkstra(G_walk, no_A_walk, c1["no_P"], weight="length")
_, rota_walk_c2 = nx.bidirectional_dijkstra(G_walk, no_A_walk, c2["no_P"], weight="length")
_, rota_walk_c3 = nx.bidirectional_dijkstra(G_walk, no_A_walk, c3["no_P"], weight="length")

# ── Caminhos de CARRO P→B para cada cenário (rede de carro) ────────────
_, rota_carro_c1 = nx.bidirectional_dijkstra(G,    c1["no_P"], no_B, weight="length")
_, rota_carro_c2 = nx.bidirectional_dijkstra(G,    c2["no_P"], no_B, weight="length")
_, rota_carro_c3 = nx.bidirectional_dijkstra(G_tr, c3["no_P"], no_B, weight="travel_time_transito")

# ── Cenário 4 — rota depende se A está ou não no grafo drive ────────────
_, rota_carro_c4 = nx.bidirectional_dijkstra(G, c4["no_P"], no_B, weight="length")
if c4["c4_caminhou"]:
    # A não estava no grafo drive: há caminhada A→P4
    _, rota_walk_c4 = nx.bidirectional_dijkstra(G_walk, no_A_walk, c4["no_P"], weight="length")
else:
    rota_walk_c4 = []  # sem caminhada

mapa = folium.Map(location=[centro[0], centro[1]], zoom_start=14, tiles="CartoDB positron")

# ── Marcadores de A e B ──────────────────────────────────────────────────
folium.Marker(
    [PONTO_A["lat"], PONTO_A["lon"]], popup=f"A – {PONTO_A['nome']}",
    icon=folium.Icon(color="green", icon="home")
).add_to(mapa)
folium.Marker(
    [PONTO_B["lat"], PONTO_B["lon"]], popup=f"B – {PONTO_B['nome']}",
    icon=folium.Icon(color="red", icon="flag")
).add_to(mapa)

# ── Marcadores dos pontos de embarque P de cada cenário ─────────────────
folium.Marker(
    [G.nodes[c1["no_P"]]["y"], G.nodes[c1["no_P"]]["x"]],
    popup=f"P¹ – Menor distância<br>{c1['walk_m']:.0f} m de caminhada",
    icon=folium.Icon(color="blue", icon="car")
).add_to(mapa)
folium.Marker(
    [G.nodes[c2["no_P"]]["y"], G.nodes[c2["no_P"]]["x"]],
    popup=f"P² – Mais rápido s/ trânsito<br>{c2['walk_m']:.0f} m de caminhada",
    icon=folium.Icon(color="purple", icon="car")
).add_to(mapa)
folium.Marker(
    [G.nodes[c3["no_P"]]["y"], G.nodes[c3["no_P"]]["x"]],
    popup=f"P³ – Mais rápido c/ trânsito<br>{c3['walk_m']:.0f} m de caminhada",
    icon=folium.Icon(color="orange", icon="car")
).add_to(mapa)

# Marcador P4 só aparece se houve caminhada forçada
if c4["c4_caminhou"] and c4["no_P"] != c1["no_P"] and c4["no_P"] != c2["no_P"]:
    folium.Marker(
        [G.nodes[c4["no_P"]]["y"], G.nodes[c4["no_P"]]["x"]],
        popup=f"P⁴ – C4 (P mais próximo)<br>{c4['walk_m']:.0f} m de caminhada",
        icon=folium.Icon(color="gray", icon="car")
    ).add_to(mapa)

# ── Raio de caminhada máxima a partir de A ───────────────────────────────
folium.Circle(
    [PONTO_A["lat"], PONTO_A["lon"]], radius=X,
    color="gray", fill=True, fill_opacity=0.07,
    tooltip=f"Raio {X} m"
).add_to(mapa)

# ── Cenário 1 — Menor distância (azul) ───────────────────────────────────
fg1 = folium.FeatureGroup(name="C1 – Menor distância", show=True)
folium.PolyLine(coords_rota(G_walk, rota_walk_c1), color="blue", weight=4,
                tooltip="C1 – Caminhada A→P¹").add_to(fg1)
folium.PolyLine(coords_rota(G, rota_carro_c1), color="blue", weight=4, dash_array="8",
                tooltip="C1 – Carro P¹→B").add_to(fg1)
fg1.add_to(mapa)

# ── Cenário 2 — Mais rápido s/ trânsito (roxo) ──────────────────────────
fg2 = folium.FeatureGroup(name="C2 – Mais rápido s/ trânsito", show=True)
folium.PolyLine(coords_rota(G_walk, rota_walk_c2), color="purple", weight=4,
                tooltip="C2 – Caminhada A→P²").add_to(fg2)
folium.PolyLine(coords_rota(G, rota_carro_c2), color="purple", weight=4, dash_array="8",
                tooltip="C2 – Carro P²→B").add_to(fg2)
fg2.add_to(mapa)

# ── Cenário 3 — Mais rápido c/ trânsito (laranja) ───────────────────────
fg3 = folium.FeatureGroup(name="C3 – Mais rápido c/ trânsito", show=True)
folium.PolyLine(coords_rota(G_walk, rota_walk_c3), color="orange", weight=4,
                tooltip="C3 – Caminhada A→P³").add_to(fg3)
folium.PolyLine(coords_rota(G_tr, rota_carro_c3), color="orange", weight=4, dash_array="8",
                tooltip="C3 – Carro P³→B (com trânsito)").add_to(fg3)
fg3.add_to(mapa)

# ── Cenário 4 — Sem caminhada (cinza, desligado por padrão) ─────────────
if c4["c4_caminhou"]:
    label_c4   = f"C4 – Sem caminhada (walk forçado {c4['walk_m']:.0f} m)"
    tooltip_walk = f"C4 – Caminhada forçada A→P⁴ ({c4['walk_m']:.0f} m)"
    tooltip_car  = "C4 – Carro P⁴→B"
else:
    label_c4   = "C4 – Sem caminhada (A→B direto)"
    tooltip_car = "C4 – A→B direto (sem caminhada)"

fg4 = folium.FeatureGroup(name=label_c4, show=False)
if c4["c4_caminhou"] and rota_walk_c4:
    folium.PolyLine(coords_rota(G_walk, rota_walk_c4), color="gray", weight=3,
                    opacity=0.7, tooltip=tooltip_walk).add_to(fg4)
folium.PolyLine(coords_rota(G, rota_carro_c4), color="gray", weight=3, opacity=0.7,
                dash_array="8", tooltip=tooltip_car).add_to(fg4)
fg4.add_to(mapa)

# ── Título flutuante + controle de camadas ──────────────────────────────
folium.map.Marker(
    [centro[0] + 0.01, centro[1]],
    icon=folium.DivIcon(html=f'<div style="font-size:16px; font-weight:bold; background:white; padding:4px 8px; border-radius:4px;">{"dijkstra bidirecional"}</div>')
).add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)

mapa